In [1]:
from pyspark.sql import SparkSession

# 1. Khởi tạo SparkSession
spark = SparkSession.builder \
    .appName("Spark_SQL_OnTap") \
    .getOrCreate()

# 2. Khai báo đường dẫn HDFS (nên có dấu / ở cuối để dễ quản lý)
HDFS_PATH = "hdfs://localhost:9000/user/bigdata/data/cleaned_dataset/"

# 3. Đọc toàn bộ dữ liệu CSV từ thư mục cleaned_dataset
df_ecom = spark.read.csv(HDFS_PATH, header=True, inferSchema=True)

# 4. Tạo Temporary View để có thể truy vấn bằng SQL
df_ecom.createOrReplaceTempView("ecom")

# 5. Kiểm tra 5 dòng đầu tiên
df_ecom.show(5)

+-------+---+------+---------+-----------+------------+-----------------+----------------+-------------------+------------+--------------+-----------+----------------+-------------------+-----------+----------------+-------------+---------------------+------------------------+-------------------+------------------------+----------------------+----------------------+--------------+---------------------------+--------------------+---------------+---------------------------+-------------------+----------------+---------------+-------------------+--------------------+---------------------------+-------------------------+----------------+-----------+----------------------------+--------------+------------------+-------------------------------+--------------------+-------------+-----------------------+-------------------+--------------------------+---------------------+----------------+-----------------+--------------------+------------------+-------------------------------+-----------------

In [12]:
df_ecom.printSchema()

root
 |-- user_id: integer (nullable = true)
 |-- age: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- country: string (nullable = true)
 |-- urban_rural: string (nullable = true)
 |-- income_level: double (nullable = true)
 |-- employment_status: string (nullable = true)
 |-- education_level: string (nullable = true)
 |-- relationship_status: string (nullable = true)
 |-- has_children: integer (nullable = true)
 |-- household_size: integer (nullable = true)
 |-- occupation: string (nullable = true)
 |-- ethnicity: string (nullable = true)
 |-- language_preference: string (nullable = true)
 |-- device_type: string (nullable = true)
 |-- weekly_purchases: integer (nullable = true)
 |-- monthly_spend: double (nullable = true)
 |-- cart_abandonment_rate: integer (nullable = true)
 |-- review_writing_frequency: integer (nullable = true)
 |-- average_order_value: integer (nullable = true)
 |-- preferred_payment_method: string (nullable = true)
 |-- coupon_usage_frequenc

In [2]:
query_1 = """
SELECT
    device_type AS Thiet_Bi,
    preferred_payment_method AS Phuong_Thuc_Thanh_Toan,
    ROUND(AVG(cart_abandonment_rate), 2) AS Ty_Le_Bo_Gio_Hang_TB
FROM ecom
WHERE income_level > (SELECT AVG(income_level) FROM ecom)
GROUP BY device_type, preferred_payment_method
ORDER BY Ty_Le_Bo_Gio_Hang_TB DESC
"""
result_1 = spark.sql(query_1)
result_1.toPandas()





,Thiet_Bi,Phuong_Thuc_Thanh_Toan,Ty_Le_Bo_Gio_Hang_TB
0,Tablet,Bank Transfer,40.79
1,Desktop,PayPal,40.45
2,Mobile,Bank Transfer,40.35
3,Tablet,Debit Card,40.35
4,Desktop,Apple Pay,40.29
5,Mobile,Debit Card,40.26
6,Desktop,Google Pay,40.24
7,Mobile,Apple Pay,40.21
8,Tablet,PayPal,40.20
9,Mobile,PayPal,40.19


In [3]:

df_ecom.select("user_id", "age", "employment_status") \
  .createOrReplaceTempView("customer_demographics")

df_ecom.select("user_id", "checkout_abandonments_per_month", "cart_items_average") \
  .createOrReplaceTempView("shopping_behaviors")


query_2_upgraded = """
WITH CombinedData AS (
    SELECT
        CASE
            WHEN d.age < 30 THEN '1. Duoi 30 tuoi'
            WHEN d.age BETWEEN 30 AND 50 THEN '2. Tu 30 den 50 tuoi'
            ELSE '3. Tren 50 tuoi'
        END AS Nhom_Tuoi,
        d.employment_status AS Tinh_Trang_Viec_Lam,
        b.checkout_abandonments_per_month,
        b.cart_items_average
    FROM customer_demographics d
    JOIN shopping_behaviors b ON d.user_id = b.user_id
)
SELECT
    Nhom_Tuoi,
    Tinh_Trang_Viec_Lam,
    SUM(checkout_abandonments_per_month) AS Tong_So_Lan_Huy_Thanh_Toan,
    ROUND(AVG(cart_items_average), 2) AS Trung_Binh_Mon_Hang_Trong_Gio
FROM CombinedData
GROUP BY
    Nhom_Tuoi,
    Tinh_Trang_Viec_Lam
ORDER BY
    Tong_So_Lan_Huy_Thanh_Toan DESC
"""
result_2 = spark.sql(query_2_upgraded)
result_2.limit(10).toPandas()

,Nhom_Tuoi,Tinh_Trang_Viec_Lam,Tong_So_Lan_Huy_Thanh_Toan,Trung_Binh_Mon_Hang_Trong_Gio
0,3. Tren 50 tuoi,Unemployed,476879,5.52
1,3. Tren 50 tuoi,Retired,476730,5.49
2,3. Tren 50 tuoi,Self-employed,475398,5.48
3,3. Tren 50 tuoi,Student,473782,5.49
4,3. Tren 50 tuoi,Employed,472162,5.49
5,2. Tu 30 den 50 tuoi,Unemployed,334662,5.50
6,2. Tu 30 den 50 tuoi,Retired,334535,5.49
7,2. Tu 30 den 50 tuoi,Self-employed,334241,5.50
8,2. Tu 30 den 50 tuoi,Student,333965,5.50
9,2. Tu 30 den 50 tuoi,Employed,333212,5.50


In [4]:
query_3 = """
WITH RankedCategories AS (
    SELECT
        product_category_preference AS Danh_Muc_San_Pham,
        country AS Quoc_Gia,
        ROUND(AVG(return_rate), 2) AS Ty_Le_Tra_Hang_TB,
        DENSE_RANK() OVER(
            PARTITION BY country
            ORDER BY AVG(return_rate) DESC
        ) AS Xep_Hang
    FROM ecom
    WHERE impulse_buying_score > 7
    GROUP BY product_category_preference, country
)
SELECT * FROM RankedCategories
WHERE Xep_Hang <= 3
ORDER BY Quoc_Gia ASC, Xep_Hang ASC
"""
result_3 = spark.sql(query_3)
result_3.toPandas()


,Danh_Muc_San_Pham,Quoc_Gia,Ty_Le_Tra_Hang_TB,Xep_Hang
0,Electronics,Australia,50.66,1
1,Fashion,Australia,50.37,2
2,Beauty,Australia,50.32,3
3,Beauty,Brazil,51.40,1
4,Books,Brazil,50.29,2
5,Electronics,Brazil,49.91,3
6,Books,Canada,50.91,1
7,Fashion,Canada,50.73,2
8,Toys,Canada,50.70,3
9,Books,China,50.79,1


In [6]:
query_4 = """
SELECT
product_category_preference AS Danh_Muc_San_Pham,
COUNT(*) AS So_Luong_Khach_Hang,
ROUND(AVG(return_rate), 2) AS Ty_Le_Hoan_Tra_TB,
ROUND(AVG(return_frequency), 2) AS Tan_Suat_Hoan_Tra_TB,
ROUND(AVG(average_order_value), 2) AS Gia_Tri_Don_Hang_TB
FROM ecom
GROUP BY
product_category_preference
HAVING
AVG(return_rate) > (
SELECT AVG(return_rate)
FROM ecom
)
ORDER BY
Ty_Le_Hoan_Tra_TB DESC
"""
result_4 = spark.sql(query_4)
result_4.toPandas()



,Danh_Muc_San_Pham,So_Luong_Khach_Hang,Ty_Le_Hoan_Tra_TB,Tan_Suat_Hoan_Tra_TB,Gia_Tri_Don_Hang_TB
0,Fashion,124904,50.06,5.99,255.26
1,Home & Kitchen,124609,50.04,6.01,254.89
2,Beauty,125409,50.03,5.99,255.77
3,Books,125054,50.02,6.00,254.94


In [7]:
query_5 = """
SELECT
shopping_time_of_day AS Khung_Gio_Mua_Sam,
CASE
WHEN weekend_shopper = 1 THEN '2. Cuoi Tuan'
ELSE '1. Ngay Thuong'
END AS Loai_Ngay,
ROUND(AVG(purchase_conversion_rate), 2) AS Ty_Le_Chuyen_Doi_TB,
SUM(ad_clicks_per_day) AS Tong_So_Click_Quang_Cao
FROM ecom
GROUP BY shopping_time_of_day, CASE WHEN weekend_shopper = 1 THEN '2. Cuoi Tuan' ELSE '1. Ngay Thuong' END
ORDER BY Ty_Le_Chuyen_Doi_TB DESC
"""
result_5 = spark.sql(query_5)
result_5.toPandas()


,Khung_Gio_Mua_Sam,Loai_Ngay,Ty_Le_Chuyen_Doi_TB,Tong_So_Click_Quang_Cao
0,Morning,2. Cuoi Tuan,50.12,311995
1,Evening,2. Cuoi Tuan,50.09,313805
2,Night,2. Cuoi Tuan,50.07,312828
3,Afternoon,1. Ngay Thuong,50.07,312977
4,Morning,1. Ngay Thuong,50.03,312894
5,Afternoon,2. Cuoi Tuan,49.96,310167
6,Night,1. Ngay Thuong,49.91,311598
7,Evening,1. Ngay Thuong,49.74,311910


In [8]:
query_6 = """
SELECT
shopping_time_of_day AS Khung_Gio_Mua_Sam,
device_type AS Loai_Thiet_Bi,
SUM(ad_views_per_day) AS Tong_Luot_Xem_Quang_Cao,
SUM(ad_clicks_per_day) AS Tong_Luot_Click_Quang_Cao,
ROUND(
SUM(ad_clicks_per_day) / SUM(ad_views_per_day) * 100,
2
) AS Ty_Le_Click_Quang_Cao,
ROUND(AVG(purchase_conversion_rate), 2) AS Ty_Le_Chuyen_Doi_TB
FROM ecom
GROUP BY
shopping_time_of_day,
device_type
HAVING
SUM(ad_views_per_day) > 0
ORDER BY
Ty_Le_Chuyen_Doi_TB DESC,
Ty_Le_Click_Quang_Cao DESC
"""
result_6 = spark.sql(query_6)
result_6.toPandas()


,Khung_Gio_Mua_Sam,Loai_Thiet_Bi,Tong_Luot_Xem_Quang_Cao,Tong_Luot_Click_Quang_Cao,Ty_Le_Click_Quang_Cao,Ty_Le_Chuyen_Doi_TB
0,Morning,Tablet,250678,62597,24.97,50.25
1,Morning,Desktop,747670,187705,25.11,50.11
2,Night,Mobile,1503556,375075,24.95,50.08
3,Afternoon,Mobile,1496748,373381,24.95,50.07
4,Morning,Mobile,1502843,374587,24.93,50.03
5,Evening,Tablet,251915,62378,24.76,50.01
6,Evening,Mobile,1499179,375251,25.03,49.95
7,Afternoon,Tablet,250294,62530,24.98,49.94
8,Afternoon,Desktop,751806,187233,24.90,49.94
9,Night,Tablet,249831,62271,24.93,49.92


In [9]:
query_7 = """
SELECT
country AS Quoc_Gia,
COUNT(*) AS So_Luong_Khach_Hang,
ROUND(AVG(ad_views_per_day), 2) AS Luot_Xem_Quang_Cao_TB,
ROUND(AVG(ad_clicks_per_day), 2) AS Luot_Click_Quang_Cao_TB,
ROUND(AVG(purchase_conversion_rate), 2) AS Ty_Le_Chuyen_Doi_TB
FROM ecom
GROUP BY
country
HAVING
AVG(ad_views_per_day) > (
SELECT AVG(ad_views_per_day)
FROM ecom
)
AND AVG(purchase_conversion_rate) < (
SELECT AVG(purchase_conversion_rate)
FROM ecom
)
ORDER BY
Luot_Xem_Quang_Cao_TB DESC,
Ty_Le_Chuyen_Doi_TB ASC
"""
result_7 = spark.sql(query_7)
result_7.toPandas()


,Quoc_Gia,So_Luong_Khach_Hang,Luot_Xem_Quang_Cao_TB,Luot_Click_Quang_Cao_TB,Ty_Le_Chuyen_Doi_TB
0,Brazil,100476,10.01,2.49,49.76
1,Japan,99144,10.00,2.50,49.88


In [11]:
query_8 = """
SELECT
social_media_influence_score AS Diem_Anh_Huong_MXH,
SUM(ad_clicks_per_day) AS Tong_Luot_Click_Quang_Cao,
ROUND(AVG(purchase_conversion_rate), 4) AS Ty_Le_Chuyen_Doi_TB
FROM ecom
GROUP BY social_media_influence_score
ORDER BY Diem_Anh_Huong_MXH DESC
"""
result_8 = spark.sql(query_8)
result_8.toPandas()


,Diem_Anh_Huong_MXH,Tong_Luot_Click_Quang_Cao,Ty_Le_Chuyen_Doi_TB
0,10,226930,50.0490
1,9,227082,49.8393
2,8,227796,49.9424
3,7,227579,50.0067
4,6,228464,49.9460
5,5,227356,49.9960
6,4,226210,50.0359
7,3,227612,49.8769
8,2,226808,50.1105
9,1,225158,49.9798


In [12]:
query_9 = """
WITH AbandonmentStats AS (
    SELECT
        product_category_preference AS Danh_Muc,
        SUM(checkout_abandonments_per_month) AS Tong_Huy_Theo_Loai
    FROM ecom
    GROUP BY product_category_preference
)
SELECT
    Danh_Muc,
    Tong_Huy_Theo_Loai,
    SUM(Tong_Huy_Theo_Loai) OVER() AS Tong_Huy_Toan_Cuc,
     ROUND(
        (Tong_Huy_Theo_Loai / SUM(Tong_Huy_Theo_Loai) OVER()) * 100, 2
    ) AS Phan_Tram_Dong_Gop,
        SUM(Tong_Huy_Theo_Loai) OVER(
        ORDER BY Tong_Huy_Theo_Loai DESC
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS Tong_Huy_Tich_Luy
FROM AbandonmentStats
ORDER BY Tong_Huy_Theo_Loai DESC
"""
result_9 = spark.sql(query_9)
result_9.toPandas()


,Danh_Muc,Tong_Huy_Theo_Loai,Tong_Huy_Toan_Cuc,Phan_Tram_Dong_Gop,Tong_Huy_Tich_Luy
0,Beauty,627673,4997523,12.56,627673
1,Electronics,626198,4997523,12.53,1253871
2,Sports,626036,4997523,12.53,1879907
3,Books,624112,4997523,12.49,2504019
4,Home & Kitchen,623522,4997523,12.48,3127541
5,Groceries,623509,4997523,12.48,3751050
6,Toys,623433,4997523,12.47,4374483
7,Fashion,623040,4997523,12.47,4997523


In [14]:
query_10 = """
WITH MonthlyStats AS (
       SELECT
        DATE_TRUNC('month', CAST(last_purchase_date AS DATE)) AS Thang_Giao_Dich,
        ROUND(AVG(cart_abandonment_rate), 4) AS Ty_Le_Huy_Theo_Thang
    FROM ecom
    WHERE last_purchase_date IS NOT NULL
    GROUP BY DATE_TRUNC('month', CAST(last_purchase_date AS DATE))
)
SELECT
    Thang_Giao_Dich,
    Ty_Le_Huy_Theo_Thang,

       LAG(Ty_Le_Huy_Theo_Thang, 1) OVER(ORDER BY Thang_Giao_Dich) AS Ty_Le_Thang_Truoc,

        ROUND(
        AVG(Ty_Le_Huy_Theo_Thang) OVER(
            ORDER BY Thang_Giao_Dich
            ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
        ), 4
    ) AS Trung_Binh_Truot_3_Thang

FROM MonthlyStats
ORDER BY Thang_Giao_Dich ASC
"""
result_10 = spark.sql(query_10)
df_result_10 = result_10.toPandas()
df_result_10


,Thang_Giao_Dich,Ty_Le_Huy_Theo_Thang,Ty_Le_Thang_Truoc,Trung_Binh_Truot_3_Thang
0,2025-01-01,40.4291,NaN,40.4291
1,2025-02-01,40.2905,40.4291,40.3598
2,2025-03-01,40.2052,40.2905,40.3083
3,2025-04-01,40.2237,40.2052,40.2398
4,2025-05-01,40.1849,40.2237,40.2046
5,2025-06-01,40.2220,40.1849,40.2102
6,2025-07-01,39.9199,40.2220,40.1089
7,2025-08-01,40.2003,39.9199,40.1141
8,2025-09-01,40.3196,40.2003,40.1466
9,2025-10-01,40.3650,40.3196,40.2950
